# Knob-tuning cost model (run_history ⨝ collected)

This notebook builds **one training table** where each row is a single experiment run:

- **Inputs (X)**: workload features + query-plan embedding (PCA) + hardware + **knob configuration**
- **Target (y)**: latency/cost (with OLTP throughput converted to latency)

Then it trains **separate models per engine** (PostgreSQL / MySQL) and evaluates:

- **Accuracy**: MAE / MAPE on latency (raw scale)
- **Tuning usefulness**: Spearman **across configurations within each workload**, plus Top-K hit-rate for finding the best config.

Why Spearman here is still relevant: in tuning you mostly care about **ranking configs** (which config is better), but we will choose feature subsets/hyperparams primarily by **accuracy**.

In [27]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

from scipy.stats import spearmanr

try:
    import lightgbm as lgb
except Exception:
    lgb = None

SEED = 42
np.random.seed(SEED)

def resolve_data_path(filename: str) -> Path:
    p1 = Path(filename)
    if p1.exists():
        return p1
    p2 = Path('cost_model_with_dann') / filename
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {filename} in . or cost_model_with_dann/")

RUN_HISTORY_CSV = resolve_data_path('cost_model_run_history.csv')
COLLECTED_CSV = resolve_data_path('cost_model_collected.csv')

print('RUN_HISTORY_CSV:', RUN_HISTORY_CSV)
print('COLLECTED_CSV  :', COLLECTED_CSV)
print('lightgbm available:', lgb is not None)

print('Ready.')

RUN_HISTORY_CSV: cost_model_run_history.csv
COLLECTED_CSV  : cost_model_collected.csv
lightgbm available: True
Ready.


## 1) Load the two datasets

- `cost_model_run_history.csv` contains **knobs** (`features.*`) and the observed target `target.cost`.
- `cost_model_collected.csv` contains **workload features** and **query-plan embedding** (`qp_emb_vector`).

In [28]:
run = pd.read_csv(RUN_HISTORY_CSV, low_memory=False)
col = pd.read_csv(COLLECTED_CSV)

print('run rows:', len(run), 'cols:', len(run.columns))
print('col rows:', len(col), 'cols:', len(col.columns))

print('run target.cost nulls:', int(run['target.cost'].isna().sum()))
print('run target.cost negatives:', int((run['target.cost'] < 0).sum()))

display(run.head(2))
display(col.head(2))

run rows: 14904 cols: 81
col rows: 191 cols: 109
run target.cost nulls: 0
run target.cost negatives: 5409


,target.cost,metadata.benchmark,metadata.db_engine,metadata.hardware,metadata.hardware_specs.cores,metadata.hardware_specs.ram_gb,metadata.hardware_specs.threads,metadata.source_run_history,metadata.workload,metadata.workload_key,...,features.temptable_max_ram,features.tmp_table_size,features.vacuum_cost_delay,features.vacuum_cost_limit,features.vacuum_cost_page_dirty,features.vacuum_cost_page_hit,features.vacuum_cost_page_miss,features.wal_buffers,features.wal_writer_delay,features.work_mem
0,4.770878,job,postgresql,hetzner-4c-8t-32gb,4,32,8,/home/E2ETune-AI4DB/data/postgresql/hetzner-4c...,job_134,data/postgresql/hetzner-4c-8t-32gb/job/job_134,...,NaN,NaN,0.0,200.0,20.0,1.0,1.0,-1.0,200.0,4096.0
1,1.655125,job,postgresql,hetzner-4c-8t-32gb,4,32,8,/home/E2ETune-AI4DB/data/postgresql/hetzner-4c...,job_134,data/postgresql/hetzner-4c-8t-32gb/job/job_134,...,NaN,NaN,100.0,9001.0,2010.0,3507.0,4705.0,20622.0,2401.0,5880768.0


,metadata.benchmark,metadata.db_engine,metadata.hardware,metadata.hardware_specs.cores,metadata.hardware_specs.ram_gb,metadata.hardware_specs.threads,metadata.source_collected_data,metadata.workload,metadata.workload_key,collected.internal_metrics.blks_hit,...,collected.workload_features.table_access_frequency.usertable,collected.workload_features.table_access_frequency.warehouse,collected.workload_features.table_access_frequency.watchlist,collected.workload_features.table_access_frequency.web_page,collected.workload_features.table_access_frequency.web_returns,collected.workload_features.table_access_frequency.web_sales,collected.workload_features.table_access_frequency.web_site,collected.workload_features.total_statements,collected.workload_features.write_count,qp_emb_vector
0,job,postgresql,hetzner-4c-8t-32gb,4,32,8,/home/E2ETune-AI4DB/data/postgresql/hetzner-4c...,job_134,data/postgresql/hetzner-4c-8t-32gb/job/job_134,16668.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,0.0,"[0.07148769497871399,0.04759959876537323,-0.05..."
1,job,postgresql,hetzner-4c-8t-32gb,4,32,8,/home/E2ETune-AI4DB/data/postgresql/hetzner-4c...,job_157,data/postgresql/hetzner-4c-8t-32gb/job/job_157,13164.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,0.0,"[0.04606808349490166,0.055484969168901443,-0.0..."


## 2) Clean target: convert OLTP throughput (negative) → latency (positive)

For OLTP benchmarks, `target.cost` is stored as **negative throughput**.
Convert to a positive latency proxy: $latency = 1/throughput = -1/cost$.
For non-OLTP benchmarks, negative values are treated as invalid and dropped.

In [29]:
run = run.copy()
run['benchmark'] = run['metadata.benchmark'].astype(str).str.strip().str.lower()

OLTP_BENCHES = {'tpcc', 'smallbank', 'ycsb', 'twitter', 'wikipedia'}

run['cost_raw'] = run['target.cost'].astype(float)

oltp_mask = run['benchmark'].isin(OLTP_BENCHES)
neg_mask = run['cost_raw'].notna() & (run['cost_raw'] < 0)

# Convert OLTP negative throughput -> positive latency
to_convert = oltp_mask & neg_mask
n_convert = int(to_convert.sum())
if n_convert:
    throughput = -run.loc[to_convert, 'cost_raw']
    valid = throughput > 0
    run.loc[to_convert & valid, 'cost_raw'] = 1.0 / throughput[valid]

# Drop remaining negatives (non-OLTP invalids, or bad OLTP entries)
before = len(run)
run = run.replace([np.inf, -np.inf], np.nan)
run = run.dropna(subset=['cost_raw']).copy()
run = run.loc[run['cost_raw'] > 0].copy()
after = len(run)

run['cost_log'] = np.log1p(run['cost_raw'].astype(np.float64))

print('Converted OLTP rows:', n_convert)
print('Rows kept:', after, '/', before)
print('cost_raw quantiles:', run['cost_raw'].quantile([0,0.5,0.9,0.99,1]).to_dict())

display(run.groupby('benchmark')['cost_raw'].agg(['count','min','median','max']).sort_values('count', ascending=False))

Converted OLTP rows: 5296
Rows kept: 14687 / 14904
cost_raw quantiles: {0.0: 2.393004038325262e-05, 0.5: 0.6447976087511051, 0.9: 6.332134138777688, 0.99: 37.39786038615705, 1.0: 112.7186926606944}


,count,min,median,max
benchmark,,,,
job,4545,0.205143,2.010910,112.718693
ssb_flat_tiny,1212,3.874951,5.320818,12.056904
tpch,1212,0.492217,2.197750,6.721395
twitter,1212,0.000024,0.000029,0.000075
wikipedia,1212,0.000187,0.000839,0.014027
ssb,1211,1.216124,3.147664,11.439282
tpcds,1211,0.062250,0.246652,0.825579
smallbank,1157,0.000028,0.000034,0.000039
tpcc,1110,0.000234,0.000921,0.020861


## 3) Parse `qp_emb_vector` and reduce dimensionality (PCA)

`qp_emb_vector` is a serialized vector per workload. We turn it into numeric columns using PCA (default 32 dims).
This gives query-plan signal without exploding feature count.

In [30]:
col = col.copy()

def _parse_vec(x):
    if pd.isna(x):
        return None
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.asarray(x, dtype=np.float32)
    s = str(x)
    try:
        arr = json.loads(s)
        return np.asarray(arr, dtype=np.float32)
    except Exception:
        return None

vecs = col['qp_emb_vector'].apply(_parse_vec)
dim = int(vecs.dropna().iloc[0].shape[0]) if vecs.notna().any() else 0
print('qp_emb_vector dim:', dim)

# Build matrix for rows that have vectors
has = vecs.notna()
X = np.vstack(vecs.loc[has].to_list()).astype(np.float32) if has.any() else np.zeros((0, 0), dtype=np.float32)

PCA_DIMS = 64  # was 32; higher usually preserves more plan signal
if X.shape[0] > 0:
    # Scale embeddings before PCA so large-magnitude dims don't dominate
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    ncomp = min(PCA_DIMS, Xs.shape[0] - 1, Xs.shape[1])
    pca = PCA(n_components=ncomp, random_state=SEED)
    Z = pca.fit_transform(Xs)
    pca_cols = [f'qp_pca_{i:02d}' for i in range(Z.shape[1])]
    col.loc[has, pca_cols] = Z
    # missing vectors -> 0s (neutral)
    col[pca_cols] = col[pca_cols].fillna(0.0)
    print('PCA explained variance (sum):', float(np.sum(pca.explained_variance_ratio_)))
else:
    pca_cols = []
    print('No qp_emb_vector rows found.')

# Keep only join keys + engineered columns + workload/internal metrics if desired
JOIN_KEYS = [
    'metadata.workload_key',
    'metadata.db_engine',
    'metadata.hardware_specs.cores',
    'metadata.hardware_specs.ram_gb',
    'metadata.hardware_specs.threads',
    'metadata.benchmark',
]

workload_cols = [c for c in col.columns if c.startswith('collected.workload_features.')]
internal_cols = [c for c in col.columns if c.startswith('collected.internal_metrics.')]

USE_INTERNAL_METRICS = False  # set True only if you will have these at inference time
kept = JOIN_KEYS + workload_cols + pca_cols + (internal_cols if USE_INTERNAL_METRICS else [])
col_small = col[kept].copy()

print('kept collected cols:', len(col_small.columns))
print('workload features:', len(workload_cols), 'internal:', len(internal_cols), 'pca:', len(pca_cols))

qp_emb_vector dim: 384
PCA explained variance (sum): 0.994975745677948
kept collected cols: 154
workload features: 84 internal: 14 pca: 64


## 4) Join: create the single training table

Join keys are `metadata.workload_key + engine + hardware specs (+ benchmark)` so we get:
**(workload, engine, hardware, knob_config) → cost** with workload/query-plan context.

In [31]:
# Prepare run-side join keys and knob columns
run_small = run.copy()
run_small['metadata.benchmark'] = run_small['metadata.benchmark'].astype(str)
run_small['metadata.db_engine'] = run_small['metadata.db_engine'].astype(str)

knob_cols = [c for c in run_small.columns if c.startswith('features.')]
meta_cols = [c for c in run_small.columns if c.startswith('metadata.')]

keep_run = JOIN_KEYS + ['cost_raw', 'cost_log', 'benchmark'] + knob_cols
run_small = run_small[keep_run].copy()

df = run_small.merge(col_small, on=JOIN_KEYS, how='left', validate='many_to_one')

match_rate = float(df[workload_cols].notna().any(axis=1).mean()) if workload_cols else float('nan')
print('joined rows:', len(df))
print('collected match rate (any workload_feature non-null):', match_rate)

# Basic cleanup: drop rows with no collected info at all (optional)
DROP_IF_NO_COLLECTED = True
if DROP_IF_NO_COLLECTED and workload_cols:
    before = len(df)
    df = df.loc[df[workload_cols].notna().any(axis=1)].copy()
    print('Dropped rows w/ no collected features:', before - len(df))

df['engine'] = df['metadata.db_engine'].astype(str).str.lower().str.strip()
df['workload_key'] = df['metadata.workload_key'].astype(str)
df['ram_gb'] = df['metadata.hardware_specs.ram_gb'].astype(int)

# Optional: normalized target per workload (stabilizes learning for tuning)
df['wl_median_cost'] = df.groupby('workload_key')['cost_raw'].transform('median').astype(np.float64)
df['cost_norm'] = (df['cost_raw'].astype(np.float64) / df['wl_median_cost']).replace([np.inf, -np.inf], np.nan)
df['cost_norm'] = df['cost_norm'].fillna(1.0)
df['cost_norm_log'] = np.log1p(df['cost_norm'].astype(np.float64))

display(df[['engine', 'benchmark', 'workload_key', 'ram_gb', 'cost_raw', 'cost_norm']].head())
print('engines:', df['engine'].value_counts().to_dict())
print('benchmarks:', df['benchmark'].value_counts().head(10).to_dict())

joined rows: 14687
collected match rate (any workload_feature non-null): 1.0
Dropped rows w/ no collected features: 0


,engine,benchmark,workload_key,ram_gb,cost_raw,cost_norm
0,postgresql,job,data/postgresql/hetzner-4c-8t-32gb/job/job_134,32,4.770878,2.941084
1,postgresql,job,data/postgresql/hetzner-4c-8t-32gb/job/job_134,32,1.655125,1.020328
2,postgresql,job,data/postgresql/hetzner-4c-8t-32gb/job/job_134,32,4.789134,2.952339
3,postgresql,job,data/postgresql/hetzner-4c-8t-32gb/job/job_134,32,1.622103,0.999971
4,postgresql,job,data/postgresql/hetzner-4c-8t-32gb/job/job_134,32,1.625723,1.002203


engines: {'postgresql': 13439, 'mysql': 1248}
benchmarks: {'job': 4545, 'tpch': 1212, 'twitter': 1212, 'ssb_flat_tiny': 1212, 'wikipedia': 1212, 'ssb': 1211, 'tpcds': 1211, 'smallbank': 1157, 'tpcc': 1110, 'ycsb': 605}


## 5) Feature engineering (knobs + workload + plan PCA + hardware)

We add a few safe, high-signal transforms:
- log1p for memory-like knobs (shared_buffers/work_mem/etc.) if present
- ratios like work_mem/shared_buffers, effective_cache_size/RAM
- limited workload×knob interactions (kept small to avoid feature explosion)

In [36]:
df = df.copy()

HW_COLS = [
    'metadata.hardware_specs.cores',
    'metadata.hardware_specs.threads',
    'metadata.hardware_specs.ram_gb',
]

PCA_COLS = [c for c in df.columns if c.startswith('qp_pca_')]
WL_COLS  = [c for c in df.columns if c.startswith('collected.workload_features.')]
IM_COLS  = [c for c in df.columns if c.startswith('collected.internal_metrics.')]
KNOB_COLS = [c for c in df.columns if c.startswith('features.')]

# log1p transforms for selected knobs (if present)
LOG_KNOBS = [
    'features.shared_buffers',
    'features.work_mem',
    'features.maintenance_work_mem',
    'features.effective_cache_size',
    'features.temp_buffers',
    'features.wal_buffers',
    'features.innodb_buffer_pool_size',
]
for c in LOG_KNOBS:
    if c in df.columns:
        df[c + '._log1p'] = np.log1p(np.maximum(df[c].astype(float).fillna(0.0), 0.0))

# ratios
if 'features.work_mem' in df.columns and 'features.shared_buffers' in df.columns:
    denom = df['features.shared_buffers'].astype(float).replace(0, np.nan)
    df['ratio.work_mem_over_shared_buffers'] = (df['features.work_mem'].astype(float) / denom).replace([np.inf, -np.inf], np.nan)

if 'features.effective_cache_size' in df.columns and 'metadata.hardware_specs.ram_gb' in df.columns:
    denom = df['metadata.hardware_specs.ram_gb'].astype(float).replace(0, np.nan)
    df['ratio.effective_cache_size_over_ram_gb'] = (df['features.effective_cache_size'].astype(float) / denom).replace([np.inf, -np.inf], np.nan)

# Cache pressure style feature: shared_buffers / RAM
if 'features.shared_buffers' in df.columns and 'metadata.hardware_specs.ram_gb' in df.columns:
    denom = df['metadata.hardware_specs.ram_gb'].astype(float).replace(0, np.nan)
    df['ratio.shared_buffers_over_ram_gb'] = (df['features.shared_buffers'].astype(float) / denom).replace([np.inf, -np.inf], np.nan)

# a few workload×knob interactions
rw = 'collected.workload_features.read_write_ratio' if 'collected.workload_features.read_write_ratio' in df.columns else None
ts = 'collected.workload_features.total_statements' if 'collected.workload_features.total_statements' in df.columns else None

sb_log = 'features.shared_buffers._log1p' if 'features.shared_buffers._log1p' in df.columns else None
wm_log = 'features.work_mem._log1p' if 'features.work_mem._log1p' in df.columns else None

if rw and sb_log:
    df['x.rw_ratio_x_shared_buffers'] = df[rw].astype(float) * df[sb_log].astype(float)
if rw and wm_log:
    df['x.rw_ratio_x_work_mem'] = df[rw].astype(float) * df[wm_log].astype(float)
if ts and wm_log:
    df['x.total_statements_x_work_mem'] = df[ts].astype(float) * df[wm_log].astype(float)

# Raw-scale interaction (sometimes helps trees more than log-scale)
if rw and 'features.shared_buffers' in df.columns:
    df['x.rw_ratio_x_shared_buffers_raw'] = df[rw].astype(float) * df['features.shared_buffers'].astype(float)

EXTRA_COLS = [c for c in df.columns if c.startswith('ratio.') or c.startswith('x.')]

FEATURE_COLS = KNOB_COLS + [c for c in df.columns if c.endswith('._log1p')] + HW_COLS + WL_COLS + PCA_COLS + EXTRA_COLS
FEATURE_COLS = list(dict.fromkeys([c for c in FEATURE_COLS if c in df.columns]))

log_feat_cols = [c for c in FEATURE_COLS if c.endswith('._log1p')]
print('Feature blocks:')
print('  knobs:', len(KNOB_COLS), 'log-knobs:', len(log_feat_cols))
print('  hardware:', len(HW_COLS), 'workload:', len(WL_COLS), 'pca:', len(PCA_COLS), 'extra:', len(EXTRA_COLS))
print('  total feature cols:', len(FEATURE_COLS))

Feature blocks:
  knobs: 78 log-knobs: 7
  hardware: 3 workload: 84 pca: 64 extra: 7
  total feature cols: 236


## 6) Metrics for tuning

We evaluate configuration selection quality per workload:
- **workload-macro Spearman** across configurations (higher is better)
- **Top-K best-config hit rate**: is the true-best config in the predicted top-K?
- Accuracy: MAE / MAPE on latency

In [25]:
def safe_spearman(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    if len(a) < 3 or np.all(a == a[0]) or np.all(b == b[0]):
        return float('nan')
    return float(spearmanr(a, b)[0])

def spearman_macro_by_workload(y_true, y_pred, workloads, min_n=10):
    vals = []
    for w in pd.unique(workloads):
        m = workloads == w
        if m.sum() < min_n:
            continue
        s = safe_spearman(y_true[m], y_pred[m])
        if not np.isnan(s):
            vals.append(s)
    return float(np.mean(vals)) if vals else float('nan'), int(len(vals))

def topk_best_hit_rate(y_true, y_pred, workloads, k=3, min_n=10):
    hits = []
    for w in pd.unique(workloads):
        m = workloads == w
        if m.sum() < min_n:
            continue
        idx_true_best = np.argmin(y_true[m])
        idx_pred_sorted = np.argsort(y_pred[m])[:k]
        hits.append(float(idx_true_best in set(idx_pred_sorted)))
    return float(np.mean(hits)) if hits else float('nan')

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    nz = y_true > 0
    mape = float(np.mean(np.abs((y_true[nz]-y_pred[nz]) / y_true[nz])) * 100.0) if nz.any() else float('nan')
    return {'mae': mae, 'mape_pct': mape}

print('Metrics ready.')

Metrics ready.


## 7) Train per engine (tuning-style split)

To reflect a real tuning loop, we do a **within-workload config holdout** split by default:
- Train on **some knob configurations** for every workload
- Test on **held-out configurations** for the same workloads

This makes Top‑K hit-rate and within-workload Spearman much more representative.

(If you want the harder "generalize to unseen workloads" test, there is a toggle in the next cell.)

In [39]:
def _split_within_workload(df_engine, test_size=0.2, random_state=SEED):
    """Within-workload config split: each workload contributes train+test rows."""
    rng = np.random.RandomState(random_state)
    tr_idx = []
    te_idx = []
    # df_engine is assumed reset_index(drop=True) so indices are positional [0..n)
    for w, idxs in df_engine.groupby('workload_key').indices.items():
        idxs = np.asarray(list(idxs), dtype=int)
        n = int(len(idxs))
        if n < 3:
            # too small to split sensibly; keep in train
            tr_idx.extend(idxs.tolist())
            continue
        rng.shuffle(idxs)
        n_test = int(np.floor(n * test_size))
        n_test = max(1, min(n - 2, n_test))  # keep >=2 train rows
        te = idxs[:n_test]
        tr = idxs[n_test:]
        te_idx.extend(te.tolist())
        tr_idx.extend(tr.tolist())
    return np.asarray(tr_idx, dtype=int), np.asarray(te_idx, dtype=int)


def train_eval_engine(df_engine, feature_cols, label=''):
    df_engine = df_engine.copy().reset_index(drop=True)
    groups = df_engine['workload_key'].to_numpy()

    # Mode knobs
    USE_NORM_TARGET = True      # train regressors on cost_norm_log instead of cost_log
    INCLUDE_RANKER = True       # add LGBMRanker (lambdarank)
    SPLIT_MODE = 'within_workload'  # 'within_workload' (tuning-style) or 'holdout_workloads' (harder)

    if SPLIT_MODE == 'holdout_workloads':
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
        tr_idx, te_idx = next(gss.split(df_engine, groups=groups))
    elif SPLIT_MODE == 'within_workload':
        tr_idx, te_idx = _split_within_workload(df_engine, test_size=0.2, random_state=SEED)
    else:
        raise ValueError(f"Unknown SPLIT_MODE={SPLIT_MODE!r}")

    tr = df_engine.iloc[tr_idx].reset_index(drop=True)
    te = df_engine.iloc[te_idx].reset_index(drop=True)

    # Keep only features present and not all-NaN in train
    cols = [c for c in feature_cols if c in tr.columns and tr[c].notna().any()]

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(tr[cols])
    X_te = imp.transform(te[cols])
    X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
    X_te = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

    # Targets
    y_tr_log = tr['cost_norm_log'].to_numpy(np.float64) if USE_NORM_TARGET else tr['cost_log'].to_numpy(np.float64)
    y_te_raw = te['cost_raw'].to_numpy(np.float64)
    wl_te = te['workload_key'].to_numpy()

    models = {
        'hgb': HistGradientBoostingRegressor(
            loss='absolute_error',
            learning_rate=0.05,
            max_depth=8,
            max_iter=500,
            random_state=SEED,
        ),
        'rf': RandomForestRegressor(
            n_estimators=600,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=SEED,
        ),
    }

    rows = []

    # ---- Regression models (absolute accuracy) ----
    for name, m in models.items():
        m.fit(X_tr, y_tr_log)
        pred_log = m.predict(X_te)

        if USE_NORM_TARGET:
            pred_norm = np.expm1(pred_log)
            pred_raw = pred_norm * te['wl_median_cost'].to_numpy(np.float64)
        else:
            pred_raw = np.expm1(pred_log)

        acc = regression_metrics(y_te_raw, pred_raw)
        sp, n_wls = spearman_macro_by_workload(y_te_raw, pred_raw, wl_te, min_n=10)
        top3 = topk_best_hit_rate(y_te_raw, pred_raw, wl_te, k=3, min_n=10)
        top5 = topk_best_hit_rate(y_te_raw, pred_raw, wl_te, k=5, min_n=10)

        rows.append({
            'engine': label,
            'model': name,
            'split_mode': SPLIT_MODE,
            'test_rows': len(te),
            'test_workloads': int(pd.Series(wl_te).nunique()),
            'mae': acc['mae'],
            'mape_pct': acc['mape_pct'],
            'wl_spearman': sp,
            'n_wls': n_wls,
            'top3_hit': top3,
            'top5_hit': top5,
            'n_features': len(cols),
        })

    # ---- Ranking model (aligned with "find best config per workload") ----
    if INCLUDE_RANKER:
        if lgb is None:
            print('LightGBM not available; skipping LGBMRanker')
        else:
            # LightGBM ranker requires group sizes and expects samples sorted by group
            tr2 = tr.copy().sort_values('workload_key').reset_index(drop=True)
            te2 = te.copy().sort_values('workload_key').reset_index(drop=True)

            X_tr2 = imp.transform(tr2[cols])
            X_te2 = imp.transform(te2[cols])
            X_tr2 = np.nan_to_num(X_tr2, nan=0.0, posinf=0.0, neginf=0.0)
            X_te2 = np.nan_to_num(X_te2, nan=0.0, posinf=0.0, neginf=0.0)

            # Wrap as DataFrame to keep feature names consistent (silences sklearn warnings)
            X_tr2 = pd.DataFrame(X_tr2, columns=cols)
            X_te2 = pd.DataFrame(X_te2, columns=cols)

            # Integer relevance labels required for lambdarank.
            # IMPORTANT: use a *small fixed number of relevance levels* for stability.
            REL_LEVELS = 16
            cost_for_rank = tr2['cost_norm'].to_numpy(np.float64) if USE_NORM_TARGET else tr2['cost_raw'].to_numpy(np.float64)
            tr2 = tr2.assign(_cost_for_rank=cost_for_rank)

            # percentile rank within workload: best=0.0, worst=1.0
            pct = tr2.groupby('workload_key')['_cost_for_rank'].rank(pct=True, method='first', ascending=True).to_numpy(np.float64)
            pct = np.clip(pct, 0.0, 1.0)

            # map to [0..REL_LEVELS-1] where higher is better
            relevance = np.floor((1.0 - pct) * (REL_LEVELS - 1)).astype(int)

            group_tr = tr2.groupby('workload_key').size().to_numpy()
            label_gain = list(range(REL_LEVELS))

            ranker = lgb.LGBMRanker(
            objective='lambdarank',
            label_gain=label_gain,
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=127,
            subsample=0.8,
            colsample_bytree=0.8,
            min_data_in_leaf=10,
            min_gain_to_split=0.0,
            force_row_wise=True,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1,
            )
            ranker.fit(X_tr2, relevance, group=group_tr)

            score = ranker.predict(X_te2)
            # Convert back to a "cost-like" quantity for our existing metrics (lower is better)
            pred_cost_like = -score

            y_te2 = te2['cost_raw'].to_numpy(np.float64)
            wl_te2 = te2['workload_key'].to_numpy()

            sp, n_wls = spearman_macro_by_workload(y_te2, pred_cost_like, wl_te2, min_n=10)
            top3 = topk_best_hit_rate(y_te2, pred_cost_like, wl_te2, k=3, min_n=10)
            top5 = topk_best_hit_rate(y_te2, pred_cost_like, wl_te2, k=5, min_n=10)

            rows.append({
                'engine': label,
                'model': 'lgbm_ranker',
                'split_mode': SPLIT_MODE,
                'test_rows': len(te2),
                'test_workloads': int(pd.Series(wl_te2).nunique()),
                'mae': float('nan'),
                'mape_pct': float('nan'),
                'wl_spearman': sp,
                'n_wls': n_wls,
                'top3_hit': top3,
                'top5_hit': top5,
                'n_features': len(cols),
            })

    return pd.DataFrame(rows).sort_values(['mape_pct', 'mae'], ascending=[True, True], na_position='last')


results = []
for eng in sorted(df['engine'].unique()):
    dfe = df.loc[df['engine'] == eng].copy()
    if len(dfe) < 200:
        print('Skipping', eng, 'not enough rows:', len(dfe))
        continue
    out = train_eval_engine(dfe, FEATURE_COLS, label=eng)
    results.append(out)

summary = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
display(summary)

if not summary.empty:
    print('Best per engine (by MAPE then MAE; ranker is evaluated via wl_spearman/topK):')
    best = summary.loc[summary.groupby('engine')['mape_pct'].idxmin()]
    display(best[['engine', 'model', 'split_mode', 'mape_pct', 'mae', 'wl_spearman', 'top3_hit', 'top5_hit', 'n_features']])


,engine,model,split_mode,test_rows,test_workloads,mae,mape_pct,wl_spearman,n_wls,top3_hit,top5_hit,n_features
0,mysql,rf,within_workload,252,24,1.359911,5.276952,0.581454,12,0.500000,0.666667,133
1,mysql,hgb,within_workload,252,24,1.734554,6.262428,0.584962,12,0.250000,0.416667,133
2,mysql,lgbm_ranker,within_workload,252,24,NaN,NaN,0.542732,12,0.500000,0.583333,133
3,postgresql,hgb,within_workload,2656,141,0.029452,2.727692,0.470341,133,0.421053,0.624060,209
4,postgresql,rf,within_workload,2656,141,0.019516,2.744950,0.436936,133,0.428571,0.556391,209
5,postgresql,lgbm_ranker,within_workload,2656,141,NaN,NaN,0.436526,133,0.481203,0.609023,209


Best per engine (by MAPE then MAE; ranker is evaluated via wl_spearman/topK):


,engine,model,split_mode,mape_pct,mae,wl_spearman,top3_hit,top5_hit,n_features
0,mysql,rf,within_workload,5.276952,1.359911,0.581454,0.500000,0.666667,133
3,postgresql,hgb,within_workload,2.727692,0.029452,0.470341,0.421053,0.624060,209


## 8) Next improvements (if you want to push accuracy further)

1) Train separate models per **RAM regime** (32GB vs 64GB) — your previous results showed a strong split.
2) Try LightGBM/XGBoost regressors (often better for tabular knobs).
3) Train a ranking model (LightGBM Ranker) to directly optimize config ordering per workload.
4) Predict **relative cost**: $cost / baseline(workload)$ if you can measure a baseline config once per workload.